In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.feature_engineering import *

In [2]:
# Loading clean dataset

df = pd.read_csv("../data/processed/merged_cleaned.csv")

print(df.shape)
df.head()

(11905, 16)


,student_id,year,day,attendance,engagement_score,difficulty,country,gender,education_level,accepted,has_disability,disability_type,completion,satisfaction,self_reported_learning,final_survey_missing
0,2024_256,2024,8,1,3.09,2,Kenya,female,phd,1,1,cognitive,1.0,3.48,2.88,0
1,2025_591,2025,2,1,2.85,2,Kenya,male,masters,1,0,none,0.0,2.33,2.69,0
2,2024_986,2024,3,1,4.57,4,Germany,male,undergrad,1,0,none,0.0,3.30,2.61,0
3,2024_1038,2024,5,1,2.86,2,India,non_binary,masters,1,0,none,1.0,3.58,3.18,0
4,2025_1096,2025,4,1,1.52,3,Kenya,female,phd,1,0,none,0.0,3.42,3.11,0


In [3]:
# student level dataset
student_df = build_student_features(df)

student_df.head()

,student_id,total_days,avg_engagement,engagement_variability,avg_difficulty,completion,satisfaction,learning,has_disability,disability_type,country,year,education_level,attendance_rate
0,2024_0,12,3.533333,0.877893,2.916667,1.0,4.82,4.68,0,none,Kenya,2024,phd,0.800000
1,2024_10,9,3.475556,0.757663,3.000000,1.0,4.08,1.74,0,none,Peru,2024,masters,0.600000
2,2024_1003,10,3.535000,1.102736,3.300000,1.0,3.14,4.31,0,none,Kenya,2024,phd,0.666667
3,2024_1005,5,3.316000,0.400600,2.800000,0.0,3.26,3.11,0,none,UK,2024,undergrad,0.333333
4,2024_1006,2,3.055000,0.671751,3.000000,0.0,1.84,4.09,1,multiple,USA,2024,masters,0.133333


In [4]:
# output
student_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1629 entries, 0 to 1628
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              1629 non-null   str    
 1   total_days              1629 non-null   int64  
 2   avg_engagement          1629 non-null   float64
 3   engagement_variability  1545 non-null   float64
 4   avg_difficulty          1629 non-null   float64
 5   completion              1629 non-null   float64
 6   satisfaction            1629 non-null   float64
 7   learning                1629 non-null   float64
 8   has_disability          1629 non-null   int64  
 9   disability_type         1629 non-null   str    
 10  country                 1629 non-null   str    
 11  year                    1629 non-null   int64  
 12  education_level         1629 non-null   str    
 13  attendance_rate         1629 non-null   float64
dtypes: float64(7), int64(3), str(4)
memory usage: 178.3

In [5]:
# basic check 
 # no breaks due to agregation
 # missing values introduced: engagement variability. Most likely emerging from early dropout
 # on day 1, leading to no data
 # all types are correct?
student_df.isnull().sum()

student_id                 0
total_days                 0
avg_engagement             0
engagement_variability    84
avg_difficulty             0
completion                 0
satisfaction               0
learning                   0
has_disability             0
disability_type            0
country                    0
year                       0
education_level            0
attendance_rate            0
dtype: int64

In [6]:
# Handle variability for students with only 1 observation
# Standard deviation is undefined for students with only one observation in feature engineering script
# These cases correspond to early dropouts.
# We set variability to 0 to indicate no observed variation.
student_df["engagement_variability"] = student_df["engagement_variability"].fillna(0)

In [7]:
student_df.isnull().sum()

student_id                0
total_days                0
avg_engagement            0
engagement_variability    0
avg_difficulty            0
completion                0
satisfaction              0
learning                  0
has_disability            0
disability_type           0
country                   0
year                      0
education_level           0
attendance_rate           0
dtype: int64

In [8]:
# adding advance features (maybe they wont be too useful)
# Early behavior may lead to future completion signaling students at risk
early_engagement = (
    df[df["day"] <= 3]
    .groupby("student_id")["engagement_score"]
    .mean()
)

student_df["early_engagement"] = student_df["student_id"].map(early_engagement)
# total active days
 # Measure persistence
# Help detect dropout


total_days = df.groupby("student_id")["day"].count()

student_df["total_days"] = student_df["student_id"].map(total_days)

# engagement trend
# Are students improving or declining
# signal for retention
engagement_trend = (
    df.groupby("student_id")
    .apply(lambda x: x["engagement_score"].iloc[-1] - x["engagement_score"].iloc[0])
)

student_df["engagement_trend"] = student_df["student_id"].map(engagement_trend)

In [9]:
# sanity check

# values outside expected ranges
# weird distributions

student_df.describe()

,total_days,avg_engagement,engagement_variability,avg_difficulty,completion,satisfaction,learning,has_disability,year,attendance_rate,early_engagement,engagement_trend
count,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000
mean,7.308165,3.406409,0.853276,2.999308,0.448128,3.365021,3.360724,0.257827,2024.508901,0.487211,3.405914,-0.076777
std,3.888124,0.448259,0.336600,0.391432,0.497455,1.032412,1.125427,0.437573,0.500074,0.259208,0.591552,1.295746
min,1.000000,1.040000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,2024.000000,0.066667,1.040000,-4.000000
25%,4.000000,3.154000,0.705183,2.777778,0.000000,2.760000,2.700000,0.000000,2024.000000,0.266667,3.010000,-0.950000
50%,7.000000,3.408333,0.885435,3.000000,0.000000,3.410000,3.460000,0.000000,2025.000000,0.466667,3.418333,0.000000
75%,10.000000,3.678182,1.053331,3.214286,1.000000,4.090000,4.200000,1.000000,2025.000000,0.666667,3.813333,0.770000
max,25.000000,5.000000,2.094946,4.000000,1.000000,5.000000,5.000000,1.000000,2025.000000,1.666667,5.000000,3.800000


In [10]:
# feature insights
# what drives completion?
# where are inequalities?
# what should we fix?

# completion vs attendance
student_df.groupby("completion")["attendance_rate"].mean()

#early engagement vs completion
student_df.groupby("completion")["early_engagement"].mean()

# disability vs engagement
student_df.groupby("has_disability")["avg_engagement"].mean()

has_disability
0    3.471100
1    3.220194
Name: avg_engagement, dtype: float64

In [11]:
# handling missing values
# Some students may drop early or  not have day 1 to day 3 data
#student_df["early_engagement"] = student_df["early_engagement"].fillna(
 #   student_df["avg_engagement"]
#)

In [12]:
# saving process dataset
student_df.to_csv("../data/processed/student_level.csv", index=False)

In [13]:
# metrics standarisation

METRIC_DEFINITIONS = {
    "completion_rate": "Proportion of students who completed the course",
    "engagement_index": "Average engagement normalized to [0,1]",
    "learning_gain": "Self-reported learning weighted by attendance",
    "equity_gap": "Difference between best and worst performing groups"
}